### gpus config: server manager


> num_replicas = total_gpus / (TP × DP × PP)      

```python
# agent_loop.py
# AgentLoopManager._initialize_llm_servers
    rollout_world_size = (
        self.config.actor_rollout_ref.rollout.tensor_model_parallel_size
        
        # 单个 rollout 实例内部的 DP   │ vLLM/SGLang 引擎内部的数据并行
        * self.config.actor_rollout_ref.rollout.data_parallel_size
        
        * self.config.actor_rollout_ref.rollout.pipeline_model_parallel_size
    )
    world_size = (
        self.worker_group.world_size
        if self.worker_group
        else self.config.trainer.n_gpus_per_node * self.config.trainer.nnodes
    )

    # 独立的 rollout server 实例数 │ 多个完全独立的推理服务  
    num_replicas = world_size // rollout_world_size

    self.server_handles = [server._server_handle for server in self.rollout_replicas]
```

- `AsyncLLMServerManager(config, server_handles)`
    - 实例化 AsyncLLMServerManager 时，会传入所有的 server_handles
- `AgentLoopWorker(config, server_handles, reward_router_address)`
    -  实例化 AgentLoopWorker 时，也需要传入 server handles
    -  init 方法内部完成 `self.server_manager = AsyncLLMServerManager(config, server_handles)`
    -  run agent loop
        -  实例化 agent loop 类时，会将 server_manager 传过去
        -  AsyncLLMServerManager 负责：
            -  负载均衡：最少请求优先（heap-based）
            -  Sticky Session：同一 request_id 路由到同一 server（LRU cache）

| 特性 | num_replicas (Replica 级 DP) | data_parallel_size (引擎内部 DP) |
| :--- | :--- | :--- |
| **独立性** | 完全独立的进程/服务 | 同一引擎内共享状态 |
| **KV Cache** | 各自独立 | 可能共享 |
| **负载均衡** | 由 AsyncLLMServerManager 分发 | 由引擎内部处理 |
| **通信开销** | 无（完全独立） | 有（需要同步） |
| **容错性** | 一个挂了不影响其他 | 一个挂了整个实例挂 |

- data_parallel_size：引擎级，紧耦合，适合需要共享 KV cache 等场景                                                                      
- num_replicas：服务级，松耦合，更灵活，更容易扩展和容错  